<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/Inference_Decoding/32_topk_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

### References
Top-K, Top-P, Temperature - [Link](https://rumn.medium.com/setting-top-k-top-p-and-temperature-in-llms-3da3a8f74832)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    # temperature, top-k filter, top-p filter, sample
    logits /= temperature

    mask_top_k = torch.cat([torch.ones(top_k), torch.zeros(len(logits) - top_k)])
    logits = logits * mask_top_k
    logits[logits == 0] = float('-inf')
    logits = torch.softmax(logits, dim=-1)

    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cum_logits = torch.cumsum(sorted_logits, dim=-1)
    mask_top_p = cum_logits <= top_p
    filtered_logits = sorted_logits[mask_top_p]
    filtered_indices = sorted_indices[mask_top_p]
    filtered_logits = filtered_logits / filtered_logits.sum()
    sampled_idx_in_filtered = torch.multinomial(filtered_logits, 1)
    final_token_idx = filtered_indices[sampled_idx_in_filtered]
    return final_token_idx


In [4]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: tensor([0])


RuntimeError: cannot sample n_sample > prob_dist.size(-1) samples without replacement

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')